# Iterative Imputation (MICE)  
https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html
## What is MICE?
- MICE stands for **Multivariate Imputation by Chained Equations**.
- It is an iterative missing value handling technique where missing values are repeatedly predicted until the difference between predicted and original values becomes minimal.

## Types of Missing Data

### 1. MCAR (Missing Completely At Random)
- No pattern exists in the missing data.
- Missingness is completely random.

### 2. MAR (Missing At Random)  
- Missingness depends on other observed columns. (Use MICE)
- Can often be predicted using other features.
- Most common type in real-world datasets. 

### 3. MNAR (Missing Not At Random)
- Missingness is related to the missing value itself.  (Don't use MICE)
- Often intentional or difficult to predict.
- Usually has weak relationships with other columns.

## Using Pandas

In [51]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [3]:
# Input cols : R&D Spend,Administration,Marketing Spend
# Output cols : Profit
df = df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [4]:
# Lets intensionly add missing values for understanding the concept 
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

In [5]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


#### Step 1 - Impute all missing values with mean of respective col

In [6]:
df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


#### Step 2 - take 1 missing value 

In [7]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


#### Step 3 - Build & train a Model using non NaN Cols TO predict that NaN value

In [8]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [9]:
y = df1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [10]:
# train model and predict NaN value 
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

array([23.14158651])

#### Step 4 - Replace that NaN using the Predicted Value

In [11]:
# we have got new value for iloc(1 row ,0 col) . thats how we have to do it for all NaN values
df1.iloc[1,0] = 23.14

In [12]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


#### Step 5 - Repeat step 2,3,4 till u find all NaN are replaced by the New values(from model)

In [13]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [14]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


In [52]:
y = df1.iloc[[0,1,2,4],1]
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

In [16]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

array([11.06331285])

In [17]:
df1.iloc[3,1] = 11.06

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [18]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [19]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [20]:
y = df1.iloc[0:4,-1]
y

21    30.0
37    20.0
2     41.0
14    26.0
Name: Marketing Spend, dtype: float64

In [21]:
# Again predict 
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

array([31.56351448])

In [22]:
# impute new value
df1.iloc[4,-1] = 31.56 
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


#### Step 6 - You Got the New Dataset 

In [23]:
# After 1st Iteration : Final data set 
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


#### Step 7 - Find the Diff between your new dataset & old dataset , ALL values should be 0 . if not repeat the whole process 

In [24]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [53]:
# we have to do these all thses steps again again till all values are 0 .
# Repeat all steps

In [26]:
# iteration 2 
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [27]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

array([23.78627207])

In [28]:
df2.iloc[1,0] = 23.78

In [29]:
df2.iloc[3,1] = np.nan
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

array([11.22020174])

In [30]:
df2.iloc[3,1] = 11.22

In [31]:
df2.iloc[4,-1] = np.nan

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

array([38.87979054])

In [32]:
df2.iloc[4,-1] = 31.56

In [33]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [34]:
# Lets check now 
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [35]:
# We got 1 missing value to 0 diff ,others have small diff left . 

In [36]:
# iteration
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [37]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

array([24.57698058])

In [38]:
df3.iloc[1,0] = 24.57

In [39]:
df3.iloc[3,1] = np.nan
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

array([11.37282844])

In [40]:
df3.iloc[3,1] = 11.37

In [41]:
df3.iloc[4,-1] = np.nan

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

array([45.53976417])

In [42]:
df3.iloc[4,-1] = 45.53 
df2.iloc[3,1] = 11.22

In [43]:
# iteration 3 result
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [44]:
# compare
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97


In [45]:
# it Got worse . we have done these only 3 times . we might havr to do these 10-15 times 

## Function for these Steps :

In [55]:
# It contains all the 7 steps which is used above ( Used The help of AI for these)

def mice_imputation(df_original, n_iterations=10, tolerance=1e-3):
    """
    Perform MICE imputation
    
    Parameters:
    -----------
    df_original : pd.DataFrame
        DataFrame with missing values
    n_iterations : int
        Maximum number of iterations
    tolerance : float
        Convergence threshold
        
    Returns:
    --------
    df_imputed : pd.DataFrame
        DataFrame with imputed values
    history : list
        List of DataFrames at each iteration
    """
    
    # Initialize with mean imputation
    df_current = df_original.copy()
    for col in df_current.columns:
        df_current[col].fillna(df_current[col].mean(), inplace=True)
        
    history = [df_current.copy()]
    
    # Store which values were originally missing
    missing_mask = df_original.isna()
    
    # Iteration loop
    for iteration in range(n_iterations):
        df_previous = df_current.copy()
        
        # Loop through each column
        for col_idx, col_name in enumerate(df_current.columns):
            
            # Check if this column has missing values
            if not missing_mask[col_name].any():
                continue
                
            # Restore NaN for this column
            df_current.loc[missing_mask[col_name], col_name] = np.nan
            
            # Get indices for training and prediction
            train_idx = ~missing_mask[col_name]
            pred_idx = missing_mask[col_name]
            
            # Prepare training data
            other_cols = [c for c in df_current.columns if c != col_name]
            X_train = df_current.loc[train_idx, other_cols]
            y_train = df_current.loc[train_idx, col_name]
            
            # Train model
            model = LinearRegression()
            model.fit(X_train, y_train)
            
            # Predict missing values
            X_pred = df_current.loc[pred_idx, other_cols]
            predictions = model.predict(X_pred)
            
            # Update dataframe
            df_current.loc[pred_idx, col_name] = predictions
            
        # Store history
        history.append(df_current.copy())
        
        # Check convergence
        difference = np.abs(df_current - df_previous)
        max_change = difference[missing_mask].max().max()
        
        print(f"Iteration {iteration + 1}: Max change = {max_change:.6f}")
        
        if max_change < tolerance:
            print(f"Converged after {iteration + 1} iterations!")
            break
            
    return df_current, history

# Run MICE
# Assuming 'df' is your pre-existing DataFrame
df_imputed, history = mice_imputation(df, n_iterations=15, tolerance=0.01)

print("\nFinal imputed dataset:")
print(df_imputed)


Iteration 1: Max change = 13.891587
Iteration 2: Max change = 7.788589
Iteration 3: Max change = 23.948920
Iteration 4: Max change = 7.612848
Iteration 5: Max change = 0.272354
Iteration 6: Max change = 0.012787
Iteration 7: Max change = 0.000597
Converged after 7 iterations!

Final imputed dataset:
    R&D Spend  Administration  Marketing Spend
21   8.000000       15.000000        30.000000
37  26.718371        5.000000        20.000000
2   15.000000       10.000000        41.000000
14  12.000000       13.022364        26.000000
44   2.000000       15.000000        70.692041


## Using Sklearn 

In [47]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
import pandas as pd

In [48]:
# Create imputer 
imputer = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=10,
    random_state=0,
    verbose=2
)

In [49]:
# Fit and transform
# Assuming 'df' is your DataFrame with missing values
df_imputed_sklearn = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns,
    index=df.index
)

[IterativeImputer] Completing matrix with shape (5, 3)
[IterativeImputer] Ending imputation round 1/10, elapsed time 0.01
[IterativeImputer] Change: 13.891586506786137, scaled tolerance: 0.041 
[IterativeImputer] Ending imputation round 2/10, elapsed time 0.02
[IterativeImputer] Change: 7.788589474908562, scaled tolerance: 0.041 
[IterativeImputer] Ending imputation round 3/10, elapsed time 0.04
[IterativeImputer] Change: 23.948920418408953, scaled tolerance: 0.041 
[IterativeImputer] Ending imputation round 4/10, elapsed time 0.05
[IterativeImputer] Change: 7.612848445225552, scaled tolerance: 0.041 
[IterativeImputer] Ending imputation round 5/10, elapsed time 0.06
[IterativeImputer] Change: 0.27235443705808393, scaled tolerance: 0.041 
[IterativeImputer] Ending imputation round 6/10, elapsed time 0.06
[IterativeImputer] Change: 0.012786770349464405, scaled tolerance: 0.041 
[IterativeImputer] Early stopping criterion reached.


In [50]:
print("Imputed using sklearn's IterativeImputer:")
print(df_imputed_sklearn)

Imputed using sklearn's IterativeImputer:
    R&D Spend  Administration  Marketing Spend
21   8.000000       15.000000        30.000000
37  26.718225        5.000000        20.000000
2   15.000000       10.000000        41.000000
14  12.000000       13.022458        26.000000
44   2.000000       15.000000        70.692637


#### This NB will act as a Personal Journal for me to refer this code in Future .     
#### Since i don't have knowledge yet to Code at this level (So i have used AI for that)